In [7]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

from pathlib import Path


pd.set_option("display.max_columns", None)



In [8]:
# Wczytanie danych
df = pd.read_csv("../data/spambase_csv.csv")
if 'class' in df.columns:
    df = df.rename(columns={'class': 'target'})

In [9]:
# Usuniecie duplikatów
df_no_exact_dupes = df.drop_duplicates()
features = df.columns.drop('target')
df_cleaned = df_no_exact_dupes.drop_duplicates(subset=features, keep=False)


In [10]:
# Rozdzielenie cech (X) i zmiennej docelowej (y)
X = df_cleaned.drop('target', axis=1)
y = df_cleaned['target']

In [11]:
# Podział na zbiór treningowy (80%) i testowy (20%)
# Parametr stratify=y gwarantuje, że proporcje spamu i hamu będą takie same w obu zbiorach
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [13]:
# Skalowanie cech za pomocą StandardScaler
scaler = StandardScaler()
# Uczymy skaler TYLKO na zbiorze treningowym (żeby model nie "podglądał" danych testowych)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Konwersja z powrotem do DataFrame dla wygody i zachowania nazw kolumn
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

# Wyświetlenie wyników podziału
print("--- PODZIAŁ DANYCH ---")
print(f"X_train (cechy do treningu): {X_train_scaled.shape}")
print(f"X_test (cechy do testów): {X_test_scaled.shape}")
print(f"y_train (etykiety do treningu): {y_train.shape}")
print(f"y_test (etykiety do testów): {y_test.shape}")

# Opcjonalnie: upewnij się, że foldery istnieją przed zapisem
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

--- PODZIAŁ DANYCH ---
X_train (cechy do treningu): (3363, 57)
X_test (cechy do testów): (841, 57)
y_train (etykiety do treningu): (3363,)
y_test (etykiety do testów): (841,)


In [14]:
# Zapisanie przetworzonych zbiorów i skalera na dysk
X_train_scaled.to_csv('data/processed/X_train.csv', index=False)
X_test_scaled.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)
joblib.dump(scaler, 'models/scaler.pkl')

print("\nDane przetworzone i zapisane w folderze 'data/processed/'. Skaler zapisany w 'models/'.")


Dane przetworzone i zapisane w folderze 'data/processed/'. Skaler zapisany w 'models/'.
